In [20]:
# ============================================================
# GOOGLE COLAB NOTEBOOK: HEALTHCARE DIET RECOMMENDATION SYSTEM
# Domain: Healthcare
# Goal: Recommend which type of diet a patient should take
# No plotting included
# ============================================================

# =========================
# 1. Import libraries
# =========================
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [21]:
# =========================
# 2. Create healthcare dataset
# =========================

np.random.seed(42)

n = 120

patient_ids = [f"P{1000+i}" for i in range(n)]
ages = np.random.randint(18, 75, n)
genders = np.random.choice(["Male", "Female"], size=n)
weights = np.random.randint(45, 110, n)
heights = np.random.randint(145, 190, n)

bmi = np.round(weights / ((heights / 100) ** 2), 2)

disease_types = np.random.choice(
    ["Diabetes", "Hypertension", "Obesity", "Heart Disease", "Thyroid", "Healthy"],
    size=n,
    p=[0.20, 0.18, 0.20, 0.12, 0.10, 0.20],
)

activity_levels = np.random.choice(["Low", "Medium", "High"], size=n, p=[0.4, 0.4, 0.2])

daily_calories = np.random.randint(1400, 3500, n)

systolic = np.random.randint(100, 170, n)
diastolic = np.random.randint(65, 110, n)
bp = [f"{s}/{d}" for s, d in zip(systolic, diastolic)]

exercise_hours = np.round(np.random.uniform(0, 10, n), 1)


def assign_diet(bmi_value, disease, activity, calories, age, sys_bp):
    if disease == "Diabetes":
        return "Low Sugar Diet"
    elif disease == "Hypertension":
        return "Low Sodium Diet"
    elif disease == "Heart Disease":
        return "Cardiac Diet"
    elif disease == "Obesity":
        return "Weight Loss Diet"
    elif bmi_value < 18.5:
        return "High Protein Diet"
    elif bmi_value >= 25 and activity == "Low":
        return "Low Carb Diet"
    elif calories > 2800 and bmi_value > 27:
        return "Calorie Deficit Diet"
    elif age > 55:
        return "Balanced Diet"
    else:
        return "Balanced Diet"


diet_recommendation = [
    assign_diet(
        bmi[i],
        disease_types[i],
        activity_levels[i],
        daily_calories[i],
        ages[i],
        systolic[i],
    )
    for i in range(n)
]

df = pd.DataFrame(
    {
        "Patient_ID": patient_ids,
        "Age": ages,
        "Gender": genders,
        "Weight_kg": weights,
        "Height_cm": heights,
        "BMI": bmi,
        "Disease_Type": disease_types,
        "Physical_Activity_Level": activity_levels,
        "Daily_Caloric_Intake": daily_calories,
        "Blood_Pressure_mmHg": bp,
        "Weekly_Exercise_Hours": exercise_hours,
        "Diet_Recommendation": diet_recommendation,
    }
)

# Introduce a few missing values intentionally for cleaning practice
df.loc[5, "Weight_kg"] = np.nan
df.loc[10, "Height_cm"] = np.nan
df.loc[15, "Physical_Activity_Level"] = np.nan
df.loc[20, "Blood_Pressure_mmHg"] = np.nan
df.loc[25, "Disease_Type"] = np.nan

print("Dataset created successfully.")
print("Shape of dataset:", df.shape)
print()
print(df.head())

Dataset created successfully.
Shape of dataset: (120, 12)

  Patient_ID  Age  Gender  Weight_kg  Height_cm    BMI   Disease_Type  \
0      P1000   56    Male       68.0      169.0  23.81       Diabetes   
1      P1001   69    Male      103.0      186.0  29.77        Healthy   
2      P1002   46  Female       76.0      175.0  24.82        Obesity   
3      P1003   32    Male       96.0      147.0  44.43        Obesity   
4      P1004   60    Male      106.0      184.0  31.31  Heart Disease   

  Physical_Activity_Level  Daily_Caloric_Intake Blood_Pressure_mmHg  \
0                  Medium                  3394              137/82   
1                    High                  3335              136/89   
2                    High                  2299              110/97   
3                  Medium                  1759             102/104   
4                     Low                  3119             132/107   

   Weekly_Exercise_Hours   Diet_Recommendation  
0                    8.9  

In [22]:
# =========================
# 3. Basic EDA
# =========================

print("First 5 rows:")
print(df.head())

print("\nDataset info:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nStatistical summary (numerical):")
print(df.describe(include=[np.number]))

print("\nStatistical summary (categorical):")
print(df.describe(include=["object"]))

print("\nDiet recommendation counts:")
print(df["Diet_Recommendation"].value_counts())

print("\nDisease type counts:")
print(df["Disease_Type"].value_counts(dropna=False))

First 5 rows:
  Patient_ID  Age  Gender  Weight_kg  Height_cm    BMI   Disease_Type  \
0      P1000   56    Male       68.0      169.0  23.81       Diabetes   
1      P1001   69    Male      103.0      186.0  29.77        Healthy   
2      P1002   46  Female       76.0      175.0  24.82        Obesity   
3      P1003   32    Male       96.0      147.0  44.43        Obesity   
4      P1004   60    Male      106.0      184.0  31.31  Heart Disease   

  Physical_Activity_Level  Daily_Caloric_Intake Blood_Pressure_mmHg  \
0                  Medium                  3394              137/82   
1                    High                  3335              136/89   
2                    High                  2299              110/97   
3                  Medium                  1759             102/104   
4                     Low                  3119             132/107   

   Weekly_Exercise_Hours   Diet_Recommendation  
0                    8.9        Low Sugar Diet  
1                    5

In [23]:
# =========================
# 4. Data cleaning
# =========================


# Split blood pressure into systolic and diastolic
def split_bp(bp_value):
    try:
        s, d = str(bp_value).split("/")
        return int(s), int(d)
    except:
        return np.nan, np.nan


df[["Systolic_BP", "Diastolic_BP"]] = df["Blood_Pressure_mmHg"].apply(
    lambda x: pd.Series(split_bp(x))
)

# Recalculate BMI where possible after cleaning
# First fill missing weight/height with median
df["Weight_kg"] = df["Weight_kg"].fillna(df["Weight_kg"].median())
df["Height_cm"] = df["Height_cm"].fillna(df["Height_cm"].median())

# Fill missing categorical values with mode
df["Disease_Type"] = df["Disease_Type"].fillna(df["Disease_Type"].mode()[0])
df["Physical_Activity_Level"] = df["Physical_Activity_Level"].fillna(
    df["Physical_Activity_Level"].mode()[0]
)

# Fill BP missing values
df["Systolic_BP"] = df["Systolic_BP"].fillna(df["Systolic_BP"].median())
df["Diastolic_BP"] = df["Diastolic_BP"].fillna(df["Diastolic_BP"].median())

# Recalculate BMI to ensure consistency
df["BMI"] = np.round(df["Weight_kg"] / ((df["Height_cm"] / 100) ** 2), 2)

# Drop original BP string column because numeric columns are better for ML
df.drop(columns=["Blood_Pressure_mmHg"], inplace=True)

print("Data cleaning completed.")
print("\nMissing values after cleaning:")
print(df.isnull().sum())

print("\nCleaned data sample:")
print(df.head())

Data cleaning completed.

Missing values after cleaning:
Patient_ID                 0
Age                        0
Gender                     0
Weight_kg                  0
Height_cm                  0
BMI                        0
Disease_Type               0
Physical_Activity_Level    0
Daily_Caloric_Intake       0
Weekly_Exercise_Hours      0
Diet_Recommendation        0
Systolic_BP                0
Diastolic_BP               0
dtype: int64

Cleaned data sample:
  Patient_ID  Age  Gender  Weight_kg  Height_cm    BMI   Disease_Type  \
0      P1000   56    Male       68.0      169.0  23.81       Diabetes   
1      P1001   69    Male      103.0      186.0  29.77        Healthy   
2      P1002   46  Female       76.0      175.0  24.82        Obesity   
3      P1003   32    Male       96.0      147.0  44.43        Obesity   
4      P1004   60    Male      106.0      184.0  31.31  Heart Disease   

  Physical_Activity_Level  Daily_Caloric_Intake  Weekly_Exercise_Hours  \
0                 

In [24]:
# =========================
# 5. Feature engineering
# =========================


# Create BMI category
def bmi_category(val):
    if val < 18.5:
        return "Underweight"
    elif val < 25:
        return "Normal"
    elif val < 30:
        return "Overweight"
    else:
        return "Obese"


df["BMI_Category"] = df["BMI"].apply(bmi_category)


# Create age group
def age_group(age):
    if age < 30:
        return "Young"
    elif age < 50:
        return "Adult"
    else:
        return "Senior"


df["Age_Group"] = df["Age"].apply(age_group)

print("Feature engineering completed.")
print(df.head())

Feature engineering completed.
  Patient_ID  Age  Gender  Weight_kg  Height_cm    BMI   Disease_Type  \
0      P1000   56    Male       68.0      169.0  23.81       Diabetes   
1      P1001   69    Male      103.0      186.0  29.77        Healthy   
2      P1002   46  Female       76.0      175.0  24.82        Obesity   
3      P1003   32    Male       96.0      147.0  44.43        Obesity   
4      P1004   60    Male      106.0      184.0  31.31  Heart Disease   

  Physical_Activity_Level  Daily_Caloric_Intake  Weekly_Exercise_Hours  \
0                  Medium                  3394                    8.9   
1                    High                  3335                    5.6   
2                    High                  2299                    7.2   
3                  Medium                  1759                    8.1   
4                     Low                  3119                    9.9   

    Diet_Recommendation  Systolic_BP  Diastolic_BP BMI_Category Age_Group  
0        

In [25]:
# =========================
# 6. Feature selection
# =========================

features = [
    "Age",
    "Gender",
    "Weight_kg",
    "Height_cm",
    "BMI",
    "Disease_Type",
    "Physical_Activity_Level",
    "Daily_Caloric_Intake",
    "Weekly_Exercise_Hours",
    "Systolic_BP",
    "Diastolic_BP",
    "BMI_Category",
    "Age_Group",
]

target = "Diet_Recommendation"

X = df[features]
y = df[target]

print("Selected features:")
print(features)

print("\nTarget column:")
print(target)

Selected features:
['Age', 'Gender', 'Weight_kg', 'Height_cm', 'BMI', 'Disease_Type', 'Physical_Activity_Level', 'Daily_Caloric_Intake', 'Weekly_Exercise_Hours', 'Systolic_BP', 'Diastolic_BP', 'BMI_Category', 'Age_Group']

Target column:
Diet_Recommendation


In [26]:
# =========================
# 7. Train-test split
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (96, 13)
Test shape: (24, 13)


In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Numerical features
numeric_features = [
    "Age",
    "Weight_kg",
    "Height_cm",
    "BMI",
    "Daily_Caloric_Intake",
    "Weekly_Exercise_Hours",
    "Systolic_BP",
    "Diastolic_BP",
]

# Categorical features
categorical_features = [
    "Gender",
    "Disease_Type",
    "Physical_Activity_Level",
    "BMI_Category",
    "Age_Group",
]

# Apply Label Encoding to categorical columns
label_encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Numerical transformer (without imputer)
numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])

# Column Transformer
preprocessor = ColumnTransformer(
    transformers=[("num", numeric_transformer, numeric_features)],
    remainder="passthrough",  # keeps label-encoded categorical columns
)

print("Preprocessing pipeline ready.")

Preprocessing pipeline ready.


In [37]:
df

,Patient_ID,Age,Gender,Weight_kg,Height_cm,BMI,Disease_Type,Physical_Activity_Level,Daily_Caloric_Intake,Weekly_Exercise_Hours,Diet_Recommendation,Systolic_BP,Diastolic_BP,BMI_Category,Age_Group
0,P1000,56,1,68.0,169.0,23.81,0,2,3394,8.9,Low Sugar Diet,137.0,82.0,0,1
1,P1001,69,1,103.0,186.0,29.77,1,0,3335,5.6,Calorie Deficit Diet,136.0,89.0,2,1
2,P1002,46,0,76.0,175.0,24.82,4,0,2299,7.2,Weight Loss Diet,110.0,97.0,0,0
3,P1003,32,1,96.0,147.0,44.43,4,2,1759,8.1,Weight Loss Diet,102.0,104.0,1,0
4,P1004,60,1,106.0,184.0,31.31,2,1,3119,9.9,Cardiac Diet,132.0,107.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,P1115,18,1,104.0,154.0,43.85,3,1,1512,0.9,Low Sodium Diet,146.0,88.0,1,2
116,P1116,61,1,107.0,149.0,48.20,4,2,1958,9.6,Weight Loss Diet,145.0,87.0,1,1
117,P1117,25,1,76.0,180.0,23.46,4,1,3448,7.6,Weight Loss Diet,111.0,72.0,0,2
118,P1118,41,0,77.0,178.0,24.30,0,2,2053,6.4,Low Sugar Diet,112.0,101.0,0,0


In [38]:
# =========================
# 9. Build classification model
# =========================

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier(n_neighbors=5)),
    ]
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print("Model training completed.")
print("Accuracy:", round(acc * 100, 2), "%")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Model training completed.
Accuracy: 33.33 %

Classification Report:

                      precision    recall  f1-score   support

       Balanced Diet       0.00      0.00      0.00         3
Calorie Deficit Diet       0.00      0.00      0.00         1
        Cardiac Diet       0.33      0.50      0.40         4
   High Protein Diet       0.00      0.00      0.00         1
       Low Carb Diet       0.00      0.00      0.00         2
     Low Sodium Diet       0.40      0.40      0.40         5
      Low Sugar Diet       0.00      0.00      0.00         2
    Weight Loss Diet       0.67      0.67      0.67         6

            accuracy                           0.33        24
           macro avg       0.17      0.20      0.18        24
        weighted avg       0.31      0.33      0.32        24



In [40]:
# =========================
# 10. Build nearest neighbor recommender
# =========================

# Fit preprocessor on full feature data
X_processed = preprocessor.fit_transform(X)

# Build nearest neighbors model
nn_model = NearestNeighbors(n_neighbors=5, metric="euclidean")
nn_model.fit(X_processed)

print("Nearest Neighbor recommendation model built successfully.")

Nearest Neighbor recommendation model built successfully.


In [48]:
# =========================
# 11. Easy recommendation function
# =========================


def recommend_diet(
    age,
    gender,
    weight_kg,
    height_cm,
    disease_type,
    physical_activity_level,
    daily_caloric_intake,
    weekly_exercise_hours,
    systolic_bp,
    diastolic_bp,
):

    bmi = round(weight_kg / ((height_cm / 100) ** 2), 2)

    if bmi < 18.5:
        bmi_cat = "Underweight"
    elif bmi < 25:
        bmi_cat = "Normal"
    elif bmi < 30:
        bmi_cat = "Overweight"
    else:
        bmi_cat = "Obese"

    if age < 30:
        age_grp = "Young"
    elif age < 50:
        age_grp = "Adult"
    else:
        age_grp = "Senior"

    new_patient = pd.DataFrame(
        [
            {
                "Age": age,
                "Gender": gender,
                "Weight_kg": weight_kg,
                "Height_cm": height_cm,
                "BMI": bmi,
                "Disease_Type": disease_type,
                "Physical_Activity_Level": physical_activity_level,
                "Daily_Caloric_Intake": daily_caloric_intake,
                "Weekly_Exercise_Hours": weekly_exercise_hours,
                "Systolic_BP": systolic_bp,
                "Diastolic_BP": diastolic_bp,
                "BMI_Category": bmi_cat,
                "Age_Group": age_grp,
            }
        ]
    )

    # Predict diet type
    predicted_diet = model.predict(new_patient)[0]

    # Find similar patients
    new_patient_processed = preprocessor.transform(new_patient)
    distances, indices = nn_model.kneighbors(new_patient_processed)

    similar_patients = df.iloc[indices[0]][
        [
            "Patient_ID",
            "Age",
            "Gender",
            "BMI",
            "Disease_Type",
            "Physical_Activity_Level",
            "Diet_Recommendation",
        ]
    ].copy()

    return predicted_diet, similar_patients

In [49]:
# =========================
# 12. Test the recommendation system
# =========================

predicted_diet, similar_patients = recommend_diet(
    age=45,
    gender="Male",
    weight_kg=88,
    height_cm=170,
    disease_type="Diabetes",
    physical_activity_level="Low",
    daily_caloric_intake=2600,
    weekly_exercise_hours=1.5,
    systolic_bp=145,
    diastolic_bp=95,
)

print("Recommended Diet for the Patient:", predicted_diet)

print("\nTop Similar Patients:")
print(similar_patients)

ValueError: could not convert string to float: 'Male'

In [7]:
class Employee:
    def __init__(self, name, age):
        self.name = name
        self.age = age


employees = [Employee("Deepak", 25), Employee("rajan", 26), Employee("raju", 27)]

for emp in employees:
    print(emp.name, emp.age)

Deepak 25
rajan 26
raju 27


In [8]:
class Student:
    def __init__(self, age):
        if age < 0:
            raise ValueError("Age cannot be negative")
        self.age = age

In [10]:
st=Student(-1)

ValueError: Age cannot be negative

In [ ]:
# what is a business analyst